In [1]:
import sys
sys.path.insert(0,'..')
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
%autoreload
import torch
import torch.nn as nn
from source.data import trainLoader
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader
from source.model import ResNextModel
from source.train import trainModel
from source.loss import BCELoss
from source.optim import RAdam
from apex import amp

In [3]:
def train(fold):
    loader = {}
    loader['image_path'] = '../../data/train/'
    loader['label_path'] = '../../data/train.csv'
    loader['fold_idx'] = fold
    train, valid = trainLoader(**loader)
    train = DataLoader(train, batch_size=12, shuffle=True, num_workers=6, drop_last=True)
    valid = DataLoader(valid, batch_size=6, shuffle=True, num_workers=6, drop_last=True)
    model = ResNextModel()
    model = model.to('cuda:0')
    optimizer = RAdam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    model, optimizer = amp.initialize(model, optimizer, opt_level="O2",keep_batchnorm_fp32=True, verbosity=0)
    schedular = StepLR(optimizer, step_size=2, gamma=0.1)
    trainer = {}
    trainer['model'] = model
    trainer['train_data'] = train
    trainer['valid_data'] = valid
    trainer['loss_fn'] = BCELoss()
    trainer['optimizer'] = optimizer
    trainer['save_path'] = '../../model/model_{}.pt'.format(fold)
    trainer['epochs'] = 3
    trainer['batch'] = 12
    trainer['scheduler'] = schedular
    trainModel(**trainer)
    model = model.cpu()
    del model
    return None

In [ ]:
train(1)

Train Images: 462237 Valid Images: 143938
Loaded pretrained weights for efficientnet-b2


 53% 245370/462225 [1:41:33<1:29:27, 40.40it/s, trn_ls=0.79674]

In [ ]:
train(2)

In [ ]:
train(3)

In [ ]:
train(4)

In [ ]:
train(5)

Train Images: 558349 Valid Images: 115909


Downloading: "http://data.lip6.fr/cadene/pretrainedmodels/se_resnext101_32x4d-3b2fe3d8.pth" to /root/.cache/torch/checkpoints/se_resnext101_32x4d-3b2fe3d8.pth
 97%|█████████▋| 182M/187M [08:54<00:17, 360kB/s] 